# ProjectPart1

Historical experiment source from the internship project. Outputs and personal runtime metadata are cleared. This version is retained for provenance; see the root README for the maintained entry points. Colab Drive paths describe the original environment and require your own matching inputs. Source: `ProjectPart1.ipynb`. No fresh GPU execution is claimed.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os



In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt

import torch

In [ ]:
print("Torch Version :", torch.__version__)

print("GPU Available :", torch.cuda.is_available())

print("GPU Name :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/ProjectPart1"

In [ ]:
if os.path.exists(DATASET_PATH):
    print("Dataset Found")
else:
    print("Dataset NOT Found")

In [ ]:
room_folders = sorted(glob.glob(os.path.join(DATASET_PATH, "Room*")))

In [ ]:
print("Total Rooms :", len(room_folders))

for room in room_folders:
    print(room)

In [ ]:
for room in room_folders:

    images = sorted(glob.glob(os.path.join(room, "*.png")))

    print(os.path.basename(room), ":", len(images), "views")

In [ ]:
sample_image = sorted(
    glob.glob(os.path.join(room_folders[0], "*.png"))
)[0]

img = Image.open(sample_image)

print(img.size)

plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))

for i, room in enumerate(room_folders):

    image_path = sorted(glob.glob(os.path.join(room, "*.png")))[0]

    img = Image.open(image_path)

    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(room))
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
total_images = 0

for room in room_folders:

    total_images += len(
        glob.glob(os.path.join(room, "*.png"))
    )

print("Rooms :", len(room_folders))
print("Total Images :", total_images)

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import clip

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

In [ ]:
model, preprocess = clip.load(
    "ViT-B/32",
    device=device
)

In [ ]:
model.eval()

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/ProjectPart1"

In [ ]:
import os
import glob

room_folders = sorted(
    glob.glob(os.path.join(DATASET_PATH, "Room*"))
)

sample_image_path = sorted(
    glob.glob(os.path.join(room_folders[0], "*.png"))
)[0]

print(sample_image_path)

In [ ]:
from PIL import Image

image = Image.open(sample_image_path)

image

In [ ]:
image_input = preprocess(image)

In [ ]:
image_input = image_input.unsqueeze(0)

In [ ]:
image_input = image_input.to(device)

In [ ]:
with torch.no_grad():

    image_embedding = model.encode_image(image_input)

In [ ]:
print(image_embedding.shape)

In [ ]:
image_embedding = image_embedding.cpu()

In [ ]:
image_embedding = image_embedding.numpy()

In [ ]:
print(image_embedding.shape)

In [ ]:
image_embedding = image_embedding.squeeze()

In [ ]:
print(image_embedding.shape)

In [ ]:
print(image_embedding[:20])

In [ ]:
embedding_norm = np.linalg.norm(image_embedding)

print(embedding_norm)

In [ ]:
image_embedding = image_embedding / np.linalg.norm(image_embedding)

In [ ]:
print(np.linalg.norm(image_embedding))

In [ ]:
room_embeddings = {}

In [ ]:
from tqdm import tqdm

for room_path in tqdm(room_folders):

    room_name = os.path.basename(room_path)

    image_paths = sorted(
        glob.glob(os.path.join(room_path, "*.png"))
    )

    room_embeddings[room_name] = []

    for image_path in image_paths:

        image = Image.open(image_path)

        image_input = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():

            embedding = model.encode_image(image_input)

        embedding = embedding.cpu().numpy().squeeze()

        embedding = embedding / np.linalg.norm(embedding)

        room_embeddings[room_name].append(embedding)

In [ ]:
print(len(room_embeddings))

In [ ]:
print(room_embeddings.keys())

In [ ]:
for room in room_embeddings:

    print(
        room,
        len(room_embeddings[room])
    )

In [ ]:
print(
    room_embeddings["Room1"][0].shape
)

In [ ]:
print(
    np.linalg.norm(
        room_embeddings["Room1"][0]
    )
)

In [ ]:
all_ok = True

for room in room_embeddings:

    for emb in room_embeddings[room]:

        if emb.shape != (512,):

            all_ok = False

print(all_ok)

In [ ]:
random_room = random.choice(
    list(room_embeddings.keys())
)

print(random_room)

print(
    room_embeddings[random_room][3][:15]
)

In [ ]:
count = 0

for room in room_embeddings:

    count += len(room_embeddings[room])

print(count)

In [ ]:
total_memory = 0

for room in room_embeddings:

    for emb in room_embeddings[room]:

        total_memory += emb.nbytes

print(
    f"{total_memory/1024:.2f} KB"
)

In [ ]:
import pickle

with open(
    "raw_clip_embeddings.pkl",
    "wb"
) as f:

    pickle.dump(room_embeddings, f)

In [ ]:
save_path = "/content/drive/MyDrive/ProjectPart1/raw_clip_embeddings.pkl"

with open(save_path, "wb") as f:

    pickle.dump(room_embeddings, f)

In [ ]:
with open(save_path, "rb") as f:

    room_embeddings = pickle.load(f)

In [ ]:
image = Image.open(image_path).convert("RGB")

In [ ]:
fused_embeddings = {}

In [ ]:
for room in room_embeddings:

    embeddings = np.stack(room_embeddings[room])

    fused = np.mean(
        embeddings,
        axis=0
    )

    fused_embeddings[room] = fused

In [ ]:
for room in fused_embeddings:

    fused_embeddings[room] = (
        fused_embeddings[room] /
        np.linalg.norm(
            fused_embeddings[room]
        )
    )

In [ ]:
print(
    fused_embeddings["Room1"].shape
)

In [ ]:
print(
    np.linalg.norm(
        fused_embeddings["Room1"]
    )
)

In [ ]:
print(len(fused_embeddings))

In [ ]:
for room in fused_embeddings:

    print(
        room,
        fused_embeddings[room][:10]
    )

In [ ]:
with open(
    "/content/drive/MyDrive/ProjectPart1/fused_clip_embeddings.pkl",
    "wb"
) as f:

    pickle.dump(
        fused_embeddings,
        f
    )

In [ ]:
with open(
    "/content/drive/MyDrive/ProjectPart1/fused_clip_embeddings.pkl",
    "rb"
) as f:

    fused_embeddings = pickle.load(f)

In [ ]:
for room in fused_embeddings:

    print(
        room,
        fused_embeddings[room].shape,
        np.linalg.norm(
            fused_embeddings[room]
        )
    )

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
room_names = sorted(fused_embeddings.keys())

print(room_names)

In [ ]:
room_names = sorted(
    fused_embeddings.keys(),
    key=lambda x: int(x.replace("Room", ""))
)

In [ ]:
print(room_names)

In [ ]:
embedding_matrix = np.stack(

    [fused_embeddings[room]

     for room in room_names]

)

In [ ]:
print(
    embedding_matrix.shape
)

In [ ]:
similarity_matrix = cosine_similarity(

    embedding_matrix

)

In [ ]:
print(
    similarity_matrix.shape
)

In [ ]:
similarity_df = pd.DataFrame(

    similarity_matrix,

    index=room_names,

    columns=room_names

)

In [ ]:
similarity_df

In [ ]:
similarity_df = similarity_df.round(4)

similarity_df

In [ ]:
csv_path = "/content/drive/MyDrive/ProjectPart1/multiview_similarity_matrix.csv"

similarity_df.to_csv(csv_path)

In [ ]:
np.diag(similarity_matrix)

In [ ]:
for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    best_match = similarities.idxmax()

    best_score = similarities.max()

    print(

        f"{room} → {best_match} ({best_score:.4f})"

    )

In [ ]:
for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    worst_match = similarities.idxmin()

    worst_score = similarities.min()

    print(

        f"{room} → {worst_match} ({worst_score:.4f})"

    )

In [ ]:
values = similarity_matrix[np.triu_indices(11,1)]

print("Mean :",values.mean())

print("Max :",values.max())

print("Min :",values.min())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.heatmap(similarity_df, annot=True, cmap="viridis")

plt.title(

    "Multi-View CLIP Room Similarity Matrix",

    fontsize=16

)

plt.xlabel("Rooms")

plt.ylabel("Rooms")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,10))

sns.heatmap(

    similarity_df,

    annot=True,

    cmap="viridis",

    fmt=".3f",

    square=True,

    linewidths=0.5

)

plt.title(

    "Multi-View CLIP Similarity Matrix"

)

plt.tight_layout()

plt.savefig(

"/content/drive/MyDrive/ProjectPart1/Multiview_Heatmap.png",

dpi=300

)

plt.close()

In [ ]:
for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    best = similarities.idxmax()

    score = similarities.max()

    print(

        f"{room}  <-->  {best} : {score:.4f}"

    )

In [ ]:
for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    worst = similarities.idxmin()

    score = similarities.min()

    print(

        f"{room}  <-->  {worst} : {score:.4f}"

    )

In [ ]:
values = similarity_matrix[np.triu_indices(len(room_names),1)]

print(

    f"Average Similarity : {values.mean():.4f}"

)

print(

    f"Maximum Similarity : {values.max():.4f}"

)

print(

    f"Minimum Similarity : {values.min():.4f}"

)

print(

    f"Standard Deviation : {values.std():.4f}"

)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(

    values,

    bins=15,

    edgecolor="black"

)

plt.title(

    "Distribution of Room Similarities"

)

plt.xlabel(

    "Cosine Similarity"

)

plt.ylabel(

    "Count"

)

plt.show()

In [ ]:
pairs = []

for i in range(len(room_names)):

    for j in range(i+1,len(room_names)):

        pairs.append(

            (

                room_names[i],

                room_names[j],

                similarity_matrix[i,j]

            )

        )

pairs = sorted(

    pairs,

    key=lambda x:x[2],

    reverse=True

)

for p in pairs[:5]:

    print(p)

In [ ]:
pairs = sorted(

    pairs,

    key=lambda x:x[2]

)

for p in pairs[:5]:

    print(p)

In [ ]:
result_df = pd.DataFrame(

    pairs,

    columns=[

        "Room1",

        "Room2",

        "Similarity"

    ]

)

result_df.to_csv(

"/content/drive/MyDrive/ProjectPart1/Room_Pair_Similarities.csv",

index=False

)